In [1]:
import pandas as pd
import numpy as np
import h5py
import matplotlib.pyplot as plt
import seaborn as sns
import geobr
import geopandas as gpd
import matplotlib as mpl

# Métodos

In [2]:
def delete_hdf_table(file_path, key):
    with pd.HDFStore(file_path, mode='a') as store:
        if f'/{key}' in store.keys():
            store.remove(f'/{key}')
            print(f"Table '{key}' removed from HDF5.")
        else:
            print(f"Table '{key}' not found in HDF5.")

def read_hdf(file_path, key):
    return pd.read_hdf(file_path, mode = 'r', key=key, encoding='utf-8')

def show_hdf_tables(file_path):
    with pd.HDFStore(file_path) as store:
        keys = store.keys()
        print(f"Current tables in {file_path}:")
        for key in keys:
            print(f"  - {key.lstrip('/')}")

def write_hdf_chunks(df, path, key, chunksize=10_000_000, mode='a'):
    with pd.HDFStore(path, mode=mode, complevel=9, complib='blosc:lz4') as store: # Open in read+write mode to append
        for i in range(0, len(df), chunksize):
            chunk = df.iloc[i:i+chunksize]
            append_mode = i != 0 # Append after the first chunk to avoid overwriting data from previous chunks
            store.append(
                key,
                chunk,
                format='table',
                data_columns=True,
                min_itemsize={'gauge_code': 20},
                encoding='utf-8',
                append=append_mode
            )
            del chunk  # free memory
            print(f"Written rows {i} to {min(i+chunksize-1, len(df))}")

def read_hdf_chunks(file_path, key, chunksize=1_000_000):
    """
    Read HDF file in chunks and concatenate into a single DataFrame.
    Only works if the HDF key is stored in 'table' format.
    
    Parameters:
    file_path (str): Path to the HDF5 file
    key (str): Key/table name to read
    chunksize (int): Number of rows per chunk
    
    Returns:
    pd.DataFrame: Concatenated DataFrame
    """
    try:
        with pd.HDFStore(file_path, mode='r+') as store:  # Changed to 'r+' (read/write) and automatically closes the store
            if store.get_storer(key).is_table:
                dfs = []
                i = 1
                for chunk in store.select(key, chunksize=chunksize):
                    dfs.append(chunk)
                    print(f"Chunk {i} with {len(chunk)} rows read.")
                    i += 1                
                if dfs:  # Check if any chunks were read
                    return pd.concat(dfs, ignore_index=True)
                else:
                    print("No data found or chunksize too large.")
                    return pd.DataFrame()
            else:
                # If not table format, read all at once
                print("Not in table format, reading all data at once.")
                return store.select(key)  # Convert to DataFrame
    except FileNotFoundError:
        print(f"File {file_path} not found.")
        return pd.DataFrame()
    except KeyError:
        print(f"Key {key} not found in file {file_path}.")
        return pd.DataFrame()
    except Exception as e:
        print(f"Error reading HDF file: {e}")
        return pd.DataFrame()

# Processing

In [3]:
cleaned_path  = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5' 
qc_path = './1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_QC.h5'

In [4]:
show_hdf_tables(cleaned_path)

Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_CLEANED.h5:
  - table_data
  - table_data_outlier_filtered
  - table_info
  - table_outlier
  - table_outlier_filter_1
  - table_outlier_filter_1_export
  - table_outlier_filter_2_export
  - table_p_availability
  - table_preclassif
  - table_q1_gaps
  - table_q2_week
  - table_q3_outliers
  - table_qc_info


In [5]:
df_info = read_hdf(cleaned_path, 'table_info')
df_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
18977,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS
18978,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
18979,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS
18980,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS


In [6]:
df_data = read_hdf_chunks(cleaned_path, 'table_data', chunksize=10_000_000)
df_data['year'] = df_data['datetime'].dt.year
df_data



# table_data_outlier_filtered

Chunk 1 with 10000000 rows read.
Chunk 2 with 10000000 rows read.
Chunk 3 with 10000000 rows read.
Chunk 4 with 10000000 rows read.
Chunk 5 with 10000000 rows read.
Chunk 6 with 10000000 rows read.
Chunk 7 with 10000000 rows read.
Chunk 8 with 10000000 rows read.
Chunk 9 with 10000000 rows read.
Chunk 10 with 10000000 rows read.
Chunk 11 with 10000000 rows read.
Chunk 12 with 10000000 rows read.
Chunk 13 with 3595839 rows read.


,gauge_code,datetime,rain_mm,year
0,110018901A,2021-01-01,6.30,2021
1,110018901A,2021-01-02,1.79,2021
2,110018901A,2021-01-03,0.00,2021
3,110018901A,2021-01-04,16.15,2021
4,110018901A,2021-01-05,1.58,2021
...,...,...,...,...
123595834,88690050,2023-12-27,0.00,2023
123595835,88690050,2023-12-28,0.00,2023
123595836,88690050,2023-12-29,2.00,2023
123595837,88690050,2023-12-30,0.00,2023


In [7]:
df_data['year'] = df_data['datetime'].dt.year
df_data

,gauge_code,datetime,rain_mm,year
0,110018901A,2021-01-01,6.30,2021
1,110018901A,2021-01-02,1.79,2021
2,110018901A,2021-01-03,0.00,2021
3,110018901A,2021-01-04,16.15,2021
4,110018901A,2021-01-05,1.58,2021
...,...,...,...,...
123595834,88690050,2023-12-27,0.00,2023
123595835,88690050,2023-12-28,0.00,2023
123595836,88690050,2023-12-29,2.00,2023
123595837,88690050,2023-12-30,0.00,2023


In [8]:
df_qc_info = pd.read_hdf(cleaned_path, key='table_qc_info')

lower_limit = 60

df_qc_info['final_classif'] = np.where(
    # (df_qc_info['q1_gaps'] < lower_limit) |
    # (df_qc_info['q2_week'] < lower_limit) |
    # (df_qc_info['q3_outliers'] < lower_limit) |
    (df_qc_info['quality_index'] < 80) |
    (df_qc_info['p_availability'] < 90) |
    (df_qc_info['preclassif'] == 'LQ') |
    (df_qc_info['final_classif'] == 'LQ'),
    'LQ',
    df_qc_info['final_classif']
)
df_qc_info

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation,year,...,consecutive_dry_days,preclassif,p_availability,q1_gaps,q2_week,q3_outliers,quality_index,quality_label,region,final_classif
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA,1961,...,275,LQ,100.00000,100.0,71.803161,98.082192,92.471338,1 - Excellent Quality,North,LQ
1,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA,1962,...,153,LQ,100.00000,100.0,69.402386,99.452055,92.213610,1 - Excellent Quality,North,LQ
2,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA,1963,...,115,,100.00000,100.0,66.650234,99.726027,91.594065,1 - Excellent Quality,North,HQ
3,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA,1964,...,145,,100.00000,100.0,87.798731,99.726776,96.881377,1 - Excellent Quality,North,HQ
4,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA,1977,...,6,LQ,6.30137,0.0,6.458565,91.304348,26.016071,5 - Very Low Quality,North,LQ
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345863,S713,MATO GROSSO DO SUL,NOVA ANDRADINA,NOVA ANDRADINA | S713,-22.078611,-53.465833,INMET,INMET,MS,2021,...,150,LQ,100.00000,100.0,62.583426,99.452055,90.508870,1 - Excellent Quality,Central-West,LQ
345864,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS,2021,...,75,,100.00000,100.0,65.175174,98.082192,90.814342,1 - Excellent Quality,Central-West,HQ
345865,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS,2021,...,76,,100.00000,100.0,78.359308,97.808219,94.041882,1 - Excellent Quality,Central-West,HQ
345866,S716,MATO GROSSO DO SUL,SANTA RITA DO PARDO,SANTA RITA DO PARDO | S716,-21.305889,-52.820375,INMET,INMET,MS,2021,...,68,,100.00000,100.0,77.324050,98.904110,94.057040,1 - Excellent Quality,Central-West,HQ


In [11]:
df_data_qc = pd.merge(df_data, df_qc_info[['gauge_code', 'year', 'final_classif']], on=['gauge_code', 'year'], how='inner')
df_data_qc = df_data_qc[df_data_qc['final_classif'] == 'HQ']
df_data_qc = df_data_qc[['gauge_code', 'datetime', 'rain_mm']]
df_data_qc

,gauge_code,datetime,rain_mm
5844,120032801A,2021-01-02,34.4
5845,120032801A,2021-01-03,0.0
5846,120032801A,2021-01-04,0.4
5847,120032801A,2021-01-05,0.0
5848,120032801A,2021-01-06,41.8
...,...,...,...
123595834,88690050,2023-12-27,0.0
123595835,88690050,2023-12-28,0.0
123595836,88690050,2023-12-29,2.0
123595837,88690050,2023-12-30,0.0


In [12]:
df_info_qc = df_info[df_info['gauge_code'].isin(df_data_qc['gauge_code'].unique())]
df_info_qc.reset_index(drop=True, inplace=True)
df_info_qc

,gauge_code,state,city,name_station,lat,long,responsible,source,state_abbreviation
0,00047000,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.650000,-47.550000,INMET,HIDROWEB,PA
1,00047002,PARÁ,SALINÓPOLIS,SALINÓPOLIS,-0.623100,-47.353600,ANA,HIDROWEB,PA
2,00047003,PARÁ,CURUÇA,CURUÇA,-0.737500,-47.853600,ANA,HIDROWEB,PA
3,00047004,PARÁ,PRIMAVERA,PRIMAVERA,-0.929400,-47.099400,ANA,HIDROWEB,PA
4,00047005,PARÁ,MARAPANIM,MARUDA,-0.633600,-47.658300,ANA,HIDROWEB,PA
...,...,...,...,...,...,...,...,...,...
17125,S711,MATO GROSSO DO SUL,LAGUNA CARAPA,LAGUNA CARAPA | S711,-22.575389,-55.160333,INMET,INMET,MS
17126,S712,MATO GROSSO DO SUL,NOVA ALVORADA DO SUL,NOVA ALVORADA DO SUL | S712,-21.450972,-54.341972,INMET,INMET,MS
17127,S714,MATO GROSSO DO SUL,PEDRO GOMES,PEDRO GOMES | S714,-18.072778,-54.548889,INMET,INMET,MS
17128,S715,MATO GROSSO DO SUL,RIBAS DO RIO PARDO,RIBAS DO RIO PARDO | S715,-20.466694,-53.763028,INMET,INMET,MS


In [ ]:
# df_info_qc.to_hdf(qc_path, key='table_info', mode='w', format='table', encoding='utf-8', index=False, complevel=9)

write_hdf_chunks(df_data_qc, qc_path, key='table_data', chunksize=10_000_000)  
write_hdf_chunks(df_info, qc_path, key='table_info')
show_hdf_tables(qc_path)


Written rows 0 to 9999999
Written rows 10000000 to 19999999
Written rows 20000000 to 29999999
Written rows 30000000 to 39999999
Written rows 40000000 to 49999999
Written rows 50000000 to 59999999
Written rows 60000000 to 69999999
Written rows 70000000 to 79999999
Written rows 80000000 to 89999999
Written rows 90000000 to 99999999
Written rows 100000000 to 106300300
Written rows 0 to 18370
Current tables in ./1 - Organized data gauge/BRAZIL/DATASETS/BRAZIL_DAILY_1961_2024_QC.h5:
  - table_data
  - table_info
